# P3: Did they train on my data? Data analysis

This notebook shows the data of the task and the validation method of the project.
All logic is in `src/eda.py` and `src/validation.py`.
The data is only on Kaggle, thus a Kaggle job runs this notebook (job `eda-notebook` in `jobs/JOBS.md`).

Colors: **blue = member**, **orange = non-member**, **green = test**. All labels come from `known.jsonl`.

Contents:

1. What is in the data?
2. How many records are members?
3. Owners: who is a victim of OpenBUMAS?
4. Does the length tell the label?
5. How did the model see a record?
6. Validation method and scores without the model
7. Examples of texts and photos
8. Checks: photos, duplicates, titles
9. Conclusions and next steps

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src import eda
from src.validation import cross_validate

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
d = eda.load()
print(len(d.known), "known records,", len(d.test), "test records, length unit:", d.unit)

## 1. What is in the data?

In [ ]:
eda.counts(d)

In [ ]:
eda.plot_counts(d);

**Conclusion:** written after the run of the notebook.

## 2. How many records are members?

In [ ]:
eda.plot_member_rate(d);

**Conclusion:** written after the run of the notebook.

## 3. Owners: who is a victim of OpenBUMAS?

An owner is the value of the field `source`. The table counts the owners of the sets for each combination of
known records, test records and known members. Forum posts and web photos are not in the table: each of these records has its own owner.

In [ ]:
eda.owner_structure(d)

In [ ]:
eda.plot_owner_members(d);

**Conclusion:** written after the run of the notebook.

## 4. Does the length tell the label?

In [ ]:
eda.plot_lengths(d);

In [ ]:
eda.length_table(d)

**Conclusion:** written after the run of the notebook.

## 5. How did the model see a record?

An attack must give the model the same token sequence as the training.
The functions below come from `model/openbumas.py` of the task.

In [ ]:
print(eda.input_format(d))

In [ ]:
display(Markdown(eda.model_card(d)))

In [ ]:
eda.model_files(d)

**Conclusion:** written after the run of the notebook.

## 6. Validation method and scores without the model

We use `known.jsonl` as the validation set. `src/validation.py` splits it into folds that copy the relation between known and test:

- Each set owner has records in known and in test. Thus the split is inside each owner, and each fold sees each owner.
  A split by owner (GroupKFold) is not correct here: it tests new owners, and test has no new set owners.
- Single records (forum posts, web photos) are split inside their source type.
- Both splits keep the member rate.

The method uses 2 folds and 5 seeds. A score function gets the `fit` part (with labels) and gives scores for the `evaluate` part.
The metric is the server function on all records of one fold together. The tables show the mean and the standard deviation over the 10 folds.

Limits of the method:

- The `fit` part has only half of the known records of an owner. For test, we use all known records. Thus the owner prior is a little weaker in the validation than for test.
- One fold has 2500 records, test has 5000. Thus the score of one fold has more noise than the score on test.

The scores below do not use the model. An attack must be better than them.

In [ ]:
rng = np.random.default_rng(0)
baselines = {
    "random": lambda fit, ev: rng.random(len(ev)),
    "length (longer = member)": lambda fit, ev: ev["length"],
    "length (shorter = member)": lambda fit, ev: -ev["length"],
    "owner prior": eda.owner_prior,
    # Records with the same score go above the threshold together. A random order inside a tie shows the effect.
    "owner prior + random order in a tie": lambda fit, ev: eda.owner_prior(fit, ev) + 1e-6 * rng.random(len(ev)),
}
results = pd.concat([cross_validate(d.known, fn).assign(score=name) for name, fn in baselines.items()])
table = results.groupby(["score", "records"], sort=False)[["TPR@5%FPR", "AUC"]].agg(["mean", "std"]).round(3)
table.loc[(slice(None), "all"), :]

In [ ]:
table

**Conclusion:** written after the run of the notebook.

## 7. Examples of texts and photos

The start of one member and one non-member text of each source type, from known.

In [ ]:
eda.text_examples(d)

Four member photos and four non-member photos of each photo source type, with their captions.

In [ ]:
eda.photo_examples(d)

## 8. Checks: photos, duplicates, titles

In [ ]:
eda.photo_checks(d)

In [ ]:
eda.duplicates(d)

In [ ]:
eda.title_at_start(d)

**Conclusion:** written after the run of the notebook.

## 9. Conclusions and next steps

**Conclusion:** written after the run of the notebook.